# Controlling an interferometer using EPICS and a stage using ophyd
awojdyla@lbl.gov, Feb 2026

In [1]:
# !pip install pyepics
# !pip install ophyd

In [2]:
#using base 3.13.5 on GW
import epics
import ophyd

## Reading the PV for the interferometer
We assume that an IOC is running for the Smaract interferomter, and that we can poll it through the network

In [3]:
epics.caget('CATERETE:PICOSCALE:POS_0')

12333091

## Moving the stage

### using the python package

In [4]:
#!pip install newportxps

### create an ophyd wrapper

In [5]:
from bluesky import RunEngine

import bluesky.plans as bp
import bluesky.plan_stubs as bps

# Create a Run Engine
RE = RunEngine()

In [6]:
from ophyd import Device, Component as Cpt, Signal
from ophyd import EpicsSignal
from ophyd.status import SubscriptionStatus
import threading
import time

class NewportXPSMotor(Device):
    # Ophyd signals
    user_readback = Cpt(Signal, value=0, kind='hinted')
    user_setpoint = Cpt(Signal, value=0, kind='normal')
    
    def __init__(self, ip_address, stage_name, username='Administrator', 
                 password='Administrator', *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        from newportxps import NewportXPS
        
        self.stage_name = stage_name
        self.group_name = stage_name.split('.')[0]  # Extract 'Group1' from 'Group1.Pos'
        self.xps = NewportXPS(ip_address, username=username, password=password)
        
        # Update initial position
        self._update_position()
        
    def _update_position(self):
        """Read current position from hardware"""
        pos = self.xps.read_stage_position(self.stage_name)
        self.user_readback.put(pos)
        return pos
    
    def _is_moving(self):
        """Check if the motor is currently moving"""
        status = self.xps.get_group_status()
        group_status = status.get(self.group_name, '')
        return 'Moving' in group_status
    
    def set(self, position, timeout=30):
        """Move to a position and return a status object"""
        
        def check_done(*, old_value, value, **kwargs):
            """Callback to check if motion is complete"""
            return not self._is_moving()
        
        # Start the move
        self.user_setpoint.put(position)
        self.xps.move_stage(self.stage_name, position)
        
        # Create a status object that monitors completion
        status = SubscriptionStatus(self.user_readback, check_done, timeout=timeout)
        
        # Start a background thread to update position during motion
        def monitor_motion():
            while not status.done:
                self._update_position()
                time.sleep(0.05)  # 50ms polling
            # Final position update
            self._update_position()
        
        thread = threading.Thread(target=monitor_motion, daemon=True)
        thread.start()
        
        return status
    
    def read(self):
        """Read current position"""
        pos = self._update_position()
        return {f'{self.name}_user_readback': {'value': pos, 'timestamp': time.time()}}
    
    def describe(self):
        """Describe the readback signal"""
        return {f'{self.name}_user_readback': {
            'source': f'XPS:{self.stage_name}',
            'dtype': 'number',
            'shape': []
        }}

In [7]:
#stock ophyd opbject for EPICS signal
interf = EpicsSignal('CATERETE:PICOSCALE:POS_0', name='interferometer_position')

#custom ophyd object
motor = NewportXPSMotor(
    ip_address='192.168.10.20',
    stage_name='Group1.Pos',
    username='Administrator',
    password='Administrator',
    name='xps_motor'
)

pos_inter_m = interf.get()*1e-12
pos_readback_m = motor.get()[0]*1e-3
print(f"Interferometer position: {pos_inter_m*1e3} mm")
print(f"Motor readback position: {pos_readback_m*1e3} mm")


Interferometer position: 0.012336519 mm
Motor readback position: 10.000001 mm


In [8]:
# from tiled.client import simple
# c = simple('../data/test')  # or just simple() for temporary/dev storage
# #ifndef 
# # c.create_container('raw', specs=['CatalogOfBlueskyRuns'])

# from bluesky_tiled_plugins import TiledWriter
# tw = TiledWriter(c['raw'])

# from bluesky import RunEngine
# RE = RunEngine()
# RE.subscribe(tw)


# RE(bp.scan([interf, motor], motor, -10, 10, 11))

## regular loop scan with ophyd objects

In [ ]:
from time import sleep
pos_xps_mm = 0

motor.set(pos_xps_mm)  # Convert from pm to mm for motor setpoint
sleep(1)  # Allow some time for the motor to move and settle

pos_ps_au = interf.get()
print(f"Set motor to {pos_xps_mm} mm, interferometer reads {pos_ps_au*1e-8} au")

Set motor to 0 mm, interferometer reads -13.98873223 au
